### Q1:

#### (a):

#### Feature Engineering:

- Identify negation words such as not, no, never
- Combine negation word with the following word (e.g., “not bad” → “NOT_bad”)
- Create new tokens to capture reversed sentiment
- Add binary feature indicating presence of negation in sentence

#### Baseline Failure:

- TF-IDF or Bag-of-Words treats “bad” as a negative word
- Ignores the negation word “not”
- Predicts negative sentiment instead of positive

In [14]:
def handle_negation(text):
    words = text.split()
    negation_words = ["not", "no", "never"]
    result = []
    negate = False
    
    for word in words:
        if word in negation_words:
            negate = True
            continue
        if negate:
            result.append("NOT_" + word)
            negate = False
        else:
            result.append(word)
    return " ".join(result)

In [15]:
handle_negation('not bad at all')

'NOT_bad at all'

### (b):

#### Feature Engineering:
- Detect contrast between positive and negative words in same sentence
- Identify punctuation patterns such as “!” indicating exaggeration
- Create features for sentiment shift (positive → negative)
- Use keyword-based flags for sarcastic cues (e.g., “wow”, “great” + negative outcome)

#### Baseline Failure:
- Model focuses on positive words like “wow” and “great”
- Fails to understand contradiction with “broke”
- Predicts positive sentiment instead of negative



In [2]:
def detect_sarcasm(text):
    if "!" in text and "broke" in text:
        return 1
    return 0

In [8]:
detect_sarcasm('Wow great! Broke on day 1')

0

### (c):

#### Feature Engineering:

- Normalize multilingual text by mapping Hindi words to English
- Create bilingual vocabulary (Hindi + English tokens)
- Apply translation or dictionary-based replacement
- Add feature indicating presence of multiple languages in a sentence

#### Baseline Failure:

- Model trained on English vocabulary does not understand Hindi words
- Words like “accha”, “bahut” are ignored or treated as unknown
- Leads to incorrect or weak sentiment prediction

In [3]:
hindi_dict = {"accha": "good", "bahut": "very", "lekin": "but"}

def normalize_code_mix(text):
    words = text.split()
    return " ".join([hindi_dict.get(w, w) for w in words])

In [9]:
normalize_code_mix('Product bahut accha hai lekin delivery late thi')

'Product very good hai but delivery late thi'

### (d):

#### Feature Engineering:

- Identify action-based keywords (e.g., returned, refunded, replaced)
- Create rule-based features for such actions
- Map actions to sentiment scores (e.g., returned → negative)
- Add binary flag indicating implicit sentiment presence

#### Baseline Failure:
- No explicit sentiment words like “good” or “bad”
- Model fails to infer meaning from action
- Predicts neutral instead of negative

In [16]:
def implicit_sentiment(text):
    if "returned" in text:
        return -1
    return 0

In [17]:
implicit_sentiment('Returned it within 2 hours')

0

### (e):

#### Feature Engineering:
- Detect comparative keywords (better, worse, more than, less than)
- Extract entities being compared (e.g., product vs Samsung)
- Create feature indicating comparative sentiment direction
- Assign polarity based on comparison (e.g., “better than” → positive)

#### Baseline Failure:

- Model treats words independently without understanding comparison
- Does not identify which product is better
- May predict neutral or incorrect sentiment

In [18]:
def comparative_feature(text):
    if "better than" in text:
        return 1
    return 0

In [19]:
comparative_feature('Way better than my previous Samsung')

1

In [20]:
def preprocess_pipeline(text):
    text = text.lower()
    text = handle_negation(text)
    text = normalize_code_mix(text)
    
    features = {
        "sarcasm": detect_sarcasm(text),
        "implicit": implicit_sentiment(text),
        "comparative": comparative_feature(text)
    }
    
    return text, features

In [21]:
preprocess_pipeline('Way better than my previous Samsung')

('way better than my previous samsung',
 {'sarcasm': 0, 'implicit': 0, 'comparative': 1})

## Q2:

### (a):

Aspect-level classification is more difficult than review-level classification because:

- Multiple sentiments in one review:
A single review can express different sentiments for different aspects of a product.
- Aspect identification required:
The model must first identify aspects such as camera, battery, support before predicting sentiment.
- Fine-grained analysis:
Instead of assigning one overall label, the model assigns sentiment to each aspect separately.
-  Context understanding:
The same sentence may contain both positive and negative opinions, requiring deeper contextual understanding.

- Example:
“Camera is good but battery is poor”
   - Camera → Positive
   - Battery → Negative

### (b):

To improve aspect-level performance, the following strategies can be applied:

- Use advanced models:
Replace traditional models (TF-IDF) with transformer-based models like BERT that capture context better.
- Better aspect extraction:
Use techniques such as keyword extraction or dependency parsing to accurately identify aspects.
- Data augmentation:
Increase training data using paraphrasing or synthetic examples to improve generalization.
- Handle linguistic patterns:
Incorporate preprocessing techniques like negation handling, sarcasm detection, and code-mixing normalization (from Q1).
- Multi-label classification:
Use models that can assign multiple sentiment labels in a single review instead of a single output.
- Domain-specific features:
Build features specific to e-commerce reviews such as product-related keywords.

### (c):

In [22]:
def extract_aspects(text):
    aspects = []
    
    if "camera" in text and "amazing" in text:
        aspects.append(("camera", "positive"))
        
    if "battery" in text and "atrocious" in text:
        aspects.append(("battery", "negative"))
        
    if "support" in text and "unhelpful" in text:
        aspects.append(("customer support", "negative"))
        
    return aspects

extract_aspects('Amazing camera quality but the battery is atrocious and customer support was unhelpful')

[('battery', 'negative'), ('customer support', 'negative')]

- The model output correctly identifies:
   - Battery → Negative
   - Customer Support → Negative
- However, it fails to extract:
   - Camera → Positive
- This happens because the approach is rule-based and depends on exact keyword matching.
- It lacks:
   - Proper preprocessing (e.g., normalization, flexible matching)
   - Contextual understanding of the sentence
- The expected output should be:
   - Camera → Positive
   - Battery → Negative
   - Customer Support → Negative
This shows that simple rule-based methods are not fully reliable and may miss valid sentiments, so more advanced techniques are needed for accurate aspect-level sentiment extraction.